# PSX SQL Database Integration

This notebook connects the cleaned Pakistan Stock Exchange (PSX) historical dataset to a MySQL database.

## Objectives

- Store cleaned PSX historical data in MySQL.
- Validate data integrity using SQL queries.
- Demonstrate SQL database integration.
- Verify the uploaded records
- Prepare a scalable database architecture for future versions of the portfolio optimization system


In [26]:
# Install the required MySQL libraries

%pip install sqlalchemy pymysql

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [27]:
# Import required libraries

import pandas as pd
from sqlalchemy import create_engine

In [28]:
# MySQL database connection details

username = "root"
password = "IamnotBot001@"
host = "localhost"
port = "3306"
database = "psx_portfolio_db"

In [29]:
from sqlalchemy.engine import URL

connection_url = URL.create(
    drivername="mysql+pymysql",
    username=username,
    password=password,
    host=host,
    port=int(port),
    database=database
)

engine = create_engine(connection_url)

with engine.connect() as connection:
    print("Successfully connected to MySQL database!")

Successfully connected to MySQL database!


In [30]:
# Load the cleaned PSX master dataset

import os

processed_folder = "../data/processed"

print(os.listdir(processed_folder))

['PSX_2015_2025_Master_Data.csv']


In [31]:
# Load the cleaned PSX dataset into a Pandas DataFrame

file_path = "../data/processed/PSX_2015_2025_Master_Data.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Total rows:", len(df))

df.head()

Dataset loaded successfully!
Total rows: 269825


,Date,Symbol,Open,High,Low,Close,Adj Close,Volume
0,2015-01-01,ABL,113.250000,113.900002,112.510002,113.550003,33.938301,283800
1,2015-01-02,ABL,113.000000,115.000000,113.000000,114.029999,34.081764,508800
2,2015-01-05,ABL,114.150002,114.500000,113.599998,114.010002,34.075790,384200
3,2015-01-06,ABL,113.800003,114.750000,113.500000,113.669998,33.974163,458500
4,2015-01-07,ABL,113.550003,114.290001,113.099998,113.599998,33.953247,240400


In [32]:
# Check dataset structure before uploading to MySQL

print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

Dataset shape: (269825, 8)

Column names:
['Date', 'Symbol', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']


In [33]:
# Upload PSX dataset to MySQL

df.to_sql(
    name="stock_prices",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=5000
)

print("PSX stock data successfully uploaded to MySQL!")

PSX stock data successfully uploaded to MySQL!


In [34]:
# Verify total rows stored in MySQL

query = """
SELECT COUNT(*) AS total_rows
FROM stock_prices;
"""

result = pd.read_sql(query, engine)

result

,total_rows
0,269825


In [35]:
# View first 10 rows stored in MySQL

query = """
SELECT *
FROM stock_prices
LIMIT 10;
"""

sample_data = pd.read_sql(query, engine)

sample_data

,Date,Symbol,Open,High,Low,Close,Adj Close,Volume
0,2015-01-01,ABL,113.250000,113.900002,112.510002,113.550003,33.938301,283800
1,2015-01-02,ABL,113.000000,115.000000,113.000000,114.029999,34.081764,508800
2,2015-01-05,ABL,114.150002,114.500000,113.599998,114.010002,34.075790,384200
3,2015-01-06,ABL,113.800003,114.750000,113.500000,113.669998,33.974163,458500
4,2015-01-07,ABL,113.550003,114.290001,113.099998,113.599998,33.953247,240400
5,2015-01-08,ABL,113.989998,113.989998,112.849998,113.050003,33.788864,354200
6,2015-01-09,ABL,113.000000,114.000000,113.000000,113.190002,33.830704,686100
7,2015-01-12,ABL,113.900002,114.800003,113.400002,114.220001,34.138565,628900
8,2015-01-13,ABL,114.139999,114.699997,113.750000,113.900002,34.042923,237000
9,2015-01-14,ABL,114.000000,114.150002,113.500000,113.730003,33.992104,447500


In [36]:
# Check MySQL table structure

query = """
DESCRIBE stock_prices;
"""

table_structure = pd.read_sql(query, engine)

table_structure

,Field,Type,Null,Key,Default,Extra
0,Date,text,YES,,None,
1,Symbol,text,YES,,None,
2,Open,double,YES,,None,
3,High,double,YES,,None,
4,Low,double,YES,,None,
5,Close,double,YES,,None,
6,Adj Close,double,YES,,None,
7,Volume,bigint,YES,,None,


In [37]:
# Check total rows stored in MySQL

query = """
SELECT COUNT(*) AS total_rows
FROM stock_prices;
"""

row_count = pd.read_sql(query, engine)

row_count

,total_rows
0,269825


In [38]:
# Convert Date column into proper datetime format

df["Date"] = pd.to_datetime(df["Date"])

print(df["Date"].dtype)

datetime64[us]


In [39]:
# Convert Date column to proper datetime format

df["Date"] = pd.to_datetime(df["Date"])

print("Date column converted successfully!")
print("New datatype:", df["Date"].dtype)

Date column converted successfully!
New datatype: datetime64[us]


In [40]:
# Upload cleaned data again with proper Date datatype

df.to_sql(
    name="stock_prices",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=5000
)

print("Updated dataset successfully saved to MySQL!")

Updated dataset successfully saved to MySQL!


In [41]:
# Verify the MySQL table structure

table_structure = pd.read_sql(
    "DESCRIBE stock_prices;",
    engine
)

table_structure

,Field,Type,Null,Key,Default,Extra
0,Date,datetime,YES,,None,
1,Symbol,text,YES,,None,
2,Open,double,YES,,None,
3,High,double,YES,,None,
4,Low,double,YES,,None,
5,Close,double,YES,,None,
6,Adj Close,double,YES,,None,
7,Volume,bigint,YES,,None,


In [42]:
# Count unique stock symbols stored in MySQL

query = """
SELECT COUNT(DISTINCT Symbol) AS total_companies
FROM stock_prices;
"""

company_count = pd.read_sql(
    query,
    engine
)

company_count

,total_companies
0,99


In [43]:
# Check missing values in the MySQL table

query = """
SELECT
    SUM(CASE WHEN Date IS NULL THEN 1 ELSE 0 END) AS missing_dates,
    SUM(CASE WHEN Symbol IS NULL THEN 1 ELSE 0 END) AS missing_symbols,
    SUM(CASE WHEN Open IS NULL THEN 1 ELSE 0 END) AS missing_open,
    SUM(CASE WHEN High IS NULL THEN 1 ELSE 0 END) AS missing_high,
    SUM(CASE WHEN Low IS NULL THEN 1 ELSE 0 END) AS missing_low,
    SUM(CASE WHEN Close IS NULL THEN 1 ELSE 0 END) AS missing_close,
    SUM(CASE WHEN `Adj Close` IS NULL THEN 1 ELSE 0 END) AS missing_adj_close,
    SUM(CASE WHEN Volume IS NULL THEN 1 ELSE 0 END) AS missing_volume
FROM stock_prices;
"""

missing_values = pd.read_sql(
    query,
    engine
)

missing_values

,missing_dates,missing_symbols,missing_open,missing_high,missing_low,missing_close,missing_adj_close,missing_volume
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [44]:
# Check duplicate stock records in the MySQL table

query = """
SELECT
    Symbol,
    Date,
    COUNT(*) AS duplicate_count
FROM stock_prices
GROUP BY
    Symbol,
    Date
HAVING COUNT(*) > 1;
"""

duplicate_records = pd.read_sql(
    query,
    engine
)

print(
    "Total duplicate records:",
    len(duplicate_records)
)

duplicate_records

Total duplicate records: 0


,Symbol,Date,duplicate_count


In [45]:
# Retrieve sample stock data from MySQL

query = """
SELECT *
FROM stock_prices
ORDER BY Date DESC
LIMIT 10;
"""

sample_data = pd.read_sql(
    query,
    engine
)

sample_data

,Date,Symbol,Open,High,Low,Close,Adj Close,Volume
0,2025-12-31,UPFL,29000.000000,29100.000000,28700.009766,28989.000000,28392.908203,58
1,2025-12-31,UBL,422.899994,427.399994,420.000000,424.589996,407.878418,1375378
2,2025-12-31,TRG,72.559998,74.290001,72.559998,72.860001,72.860001,6847550
3,2025-12-31,TELE,11.510000,11.550000,11.280000,11.330000,11.330000,5339075
4,2025-12-31,THCCL,82.980003,85.940002,82.250000,84.180000,84.180000,5423547
5,2025-12-31,UNITY,21.900000,21.900000,21.110001,21.260000,21.260000,3943299
6,2025-12-31,THALL,521.200012,550.000000,521.000000,541.960022,529.604553,83602
7,2025-12-31,PRL,37.740002,37.740002,36.520000,36.639999,36.639999,6821326
8,2025-12-31,POL,606.000000,609.950012,606.000000,608.510010,583.847473,75207
9,2025-12-31,OGDC,280.890015,283.940002,279.000000,281.089996,274.172668,5922492


In [46]:
# Retrieve MEBL stock data from MySQL

query = """
SELECT *
FROM stock_prices
WHERE Symbol = 'MEBL'
ORDER BY Date DESC
LIMIT 10;
"""

mebl_data = pd.read_sql(
    query,
    engine
)

mebl_data

,Date,Symbol,Open,High,Low,Close,Adj Close,Volume
0,2025-12-31,MEBL,448.000000,449.899994,443.500000,444.380005,430.646698,1226754
1,2025-12-30,MEBL,444.989990,448.700012,444.000000,447.100006,433.282654,2979718
2,2025-12-29,MEBL,442.500000,444.880005,440.119995,444.000000,430.278442,1758817
3,2025-12-26,MEBL,440.000000,443.000000,439.049988,442.010010,428.349945,1785462
4,2025-12-24,MEBL,441.000000,442.000000,439.500000,439.920013,426.324524,1646026
5,2025-12-23,MEBL,440.000000,443.850006,440.000000,440.709991,427.090118,1737303
6,2025-12-22,MEBL,438.750000,443.500000,435.250000,439.459991,425.878723,1950734
7,2025-12-19,MEBL,436.489990,444.100006,435.559998,438.779999,425.219757,14227920
8,2025-12-18,MEBL,431.019989,435.989990,431.000000,434.630005,421.197998,5916732
9,2025-12-17,MEBL,432.500000,433.000000,427.000000,429.899994,416.614166,1521535


## Database Integration Summary

The cleaned PSX historical dataset was successfully integrated with the MySQL database.

### Results

- Total stock records stored: **269,825**
- Total listed companies: **99**
- Missing values: **0**
- Duplicate stock-date records: **0**
- Date column converted to the appropriate datetime format
- Python-to-MySQL connection tested successfully
- SQL data retrieval verified successfully

Note: The current version of the portfolio optimization model uses the cleaned master CSV dataset for analysis. The MySQL database has been integrated as a scalable data storage layer for future enhancements such as automatic market data updates, larger datasets, and database-driven retrieval.

